In [99]:
from pydantic_magic import pydantic_variant
from pydantic import BaseModel, Field
from typing import Literal

In [100]:
@pydantic_variant(annotations=Field(discriminator="type"))
class Test0(BaseModel):
    type: str
    test0: str = ""

class Test1(Test0):
    type: Literal["test1"] = "test1"
    test1: str = ""

class Test2(Test0):
    type: Literal["test2"] = "test2"
    test2: str = ""

class Test3(Test2):
    type: Literal["test3"] = "test3"
    test3: str = ""

pydantic_variant_init_subclass: <class '__main__.Test1'> | () | {}
pydantic_variant_init_subclass: False
::DEBUG:: typing.Annotated[NoneType, FieldInfo(annotation=NoneType, required=True, discriminator='type')] | <class '__main__.Test1'> | <class 'NoneType'>
pydantic_variant_init_subclass: <class '__main__.Test2'> | () | {}
pydantic_variant_init_subclass: False
::DEBUG:: typing.Annotated[__main__.Test1, FieldInfo(annotation=NoneType, required=True, discriminator='type')] | <class '__main__.Test2'> | <class '__main__.Test1'>
pydantic_variant_init_subclass: <class '__main__.Test3'> | () | {}
pydantic_variant_init_subclass: False
::DEBUG:: typing.Annotated[typing.Union[__main__.Test2, __main__.Test1], FieldInfo(annotation=NoneType, required=True, discriminator='type')] | <class '__main__.Test3'> | typing.Union[__main__.Test2, __main__.Test1]


In [101]:
t = Test0(type="test0")
print(t)
print(isinstance(t, Test0))
print(isinstance(t, Test1))
print(isinstance(t, Test2))
print(isinstance(t, Test3))

ValidationError: 1 validation error for RootModel[Annotated[Union[Test3, Test2, Test1], FieldInfo(annotation=NoneType, required=True, discriminator='type')]]
  Input tag 'test0' found using 'type' does not match any of the expected tags: 'test3', 'test2', 'test1' [type=union_tag_invalid, input_value={'type': 'test0'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/union_tag_invalid

In [ ]:
from typing import Annotated

Annotated[int | str, "hello"](3)

TypeError: 'types.UnionType' object is not callable

In [ ]:

from typing import Literal
from pydantic import BaseModel, Field
from pydantic_magic import pydantic_variant

@pydantic_variant(annotations=Field(discriminator="name"))
class Fruit(BaseModel):
    name: str

class Apple(Fruit):
    name: Literal["apple"] = "apple"

class Orange(Fruit):
    name: Literal["orange"] = "orange"

class Cherry(Fruit):
    name: Literal["cherry"] = "cherry"


Fruit()
assert isinstance(Fruit(name="apple"), Apple)
assert isinstance(Fruit(name="orange"), Orange)
assert isinstance(Fruit(name="cherry"), Cherry)
assert Apple().name == "apple"
assert Orange().name == "orange"
assert Cherry().name == "cherry"

pydantic_variant_init_subclass: <class '__main__.Apple'> | () | {}
pydantic_variant_init_subclass: typing.Annotated[NoneType, FieldInfo(annotation=NoneType, required=True, discriminator='name')]
pydantic_variant_init_subclass: typing.Annotated[typing.Optional[__main__.Apple], FieldInfo(annotation=NoneType, required=True, discriminator='name')]
pydantic_variant_init_subclass: <class '__main__.Orange'> | () | {}
pydantic_variant_init_subclass: typing.Annotated[typing.Optional[__main__.Apple], FieldInfo(annotation=NoneType, required=True, discriminator='name')]
pydantic_variant_init_subclass: typing.Annotated[typing.Union[__main__.Orange, __main__.Apple, NoneType], FieldInfo(annotation=NoneType, required=True, discriminator='name')]
pydantic_variant_init_subclass: <class '__main__.Cherry'> | () | {}
pydantic_variant_init_subclass: typing.Annotated[typing.Union[__main__.Orange, __main__.Apple, NoneType], FieldInfo(annotation=NoneType, required=True, discriminator='name')]
pydantic_variant_

ValidationError: 1 validation error for RootModel[Annotated[Union[Cherry, Orange, Apple, NoneType], FieldInfo(annotation=NoneType, required=True, discriminator='name')]]
  Unable to extract tag using discriminator 'name' [type=union_tag_not_found, input_value=PydanticUndefined, input_type=PydanticUndefinedType]
    For further information visit https://errors.pydantic.dev/2.10/v/union_tag_not_found

In [108]:

import math
from abc import abstractmethod
from pydantic import BaseModel, model_validator
from pydantic_magic import pydantic_variant

@pydantic_variant(annotations=Field(discriminator="type"))
@pydantic_variant
class Shape(BaseModel):
    type: str

    @abstractmethod
    def area(self) -> float: ...

    @abstractmethod
    def perimeter(self) -> float: ...

class Polygon(Shape):
    type: Literal["polygon"] = "polygon"

class Rectangle(Polygon):
    type: Literal["rectangle"] = "rectangle"

    length: float
    width: float

    def area(self) -> float:
        return self.length * self.width

    def perimeter(self) -> float:
        return 2 * self.length + 2 * self.width

class Square(Rectangle):
    type: Literal["square"] = "square"

    @model_validator(mode="before")
    def specify_one_length_width(cls, v: Any) -> Any:
        if not isinstance(v, dict):
            return v

        length = v.get("length")
        width = v.get("width")

        if length is not None:
            v["length"] = v["width"] = length
        elif width is not None:
            v["length"] = v["width"] = width
        else:
            raise ValueError("must specify one of 'length' or 'width'")

        return v

class Circle(Shape):
    type: Literal["circle"] = "circle"

    radius: float

    def area(self) -> float:
        return math.pi * self.radius**2

    def perimeter(self) -> float:
        return 2 * math.pi * self.radius

pydantic_variant_init_subclass: <class '__main__.Polygon'> | () | {}
pydantic_variant_init_subclass: True
pydantic_variant_init_subclass: <class '__main__.Rectangle'> | () | {}
pydantic_variant_init_subclass: False
::DEBUG:: typing.Annotated[NoneType, FieldInfo(annotation=NoneType, required=True, discriminator='type')] / <class '__main__.Rectangle'> / <class 'NoneType'>
::DEBUG:: typing.Annotated[NoneType, FieldInfo(annotation=NoneType, required=True, discriminator='type')] / <class '__main__.Rectangle'> / <class '__main__.Rectangle'>
pydantic_variant_init_subclass: <class '__main__.Square'> | () | {}
pydantic_variant_init_subclass: False
::DEBUG:: typing.Annotated[__main__.Rectangle, FieldInfo(annotation=NoneType, required=True, discriminator='type')] / <class '__main__.Square'> / <class '__main__.Rectangle'>
::DEBUG:: typing.Annotated[__main__.Rectangle, FieldInfo(annotation=NoneType, required=True, discriminator='type')] / <class '__main__.Square'> / typing.Union[__main__.Square, __

In [110]:


# Shape()
#> ValidationError: Unable to extract tag using discriminator 'type'

# Shape(type="unknown")
#> ValidationError: Discriminator 'type' does not match any of the expected tags: 'circle', 'square', 'rectangle', 'polygon'

Shape(type="polygon")
#> TypeError: Can't instantiate abstract class Polygon without an implementation for abstract methods 'area', 'perimeter'

# Shape(type="rectangle")
#> ValidationError: Missing `rectangle.length` and `rectangle.width`

rectangle = Shape(type="rectangle", length=1.0, width=2.0)
assert type(rectangle) is Rectangle
assert isinstance(rectangle, Shape)
assert isinstance(rectangle, Polygon)
assert isinstance(rectangle, Rectangle)
assert not isinstance(rectangle, Square)
assert not isinstance(rectangle, Circle)
assert rectangle.area() == 2.0
assert rectangle.perimeter() == 6.0

square = Shape(type="square", length=1.0)
assert type(square) is Square
assert isinstance(square, Shape)
assert isinstance(square, Polygon)
assert isinstance(square, Rectangle)
assert isinstance(square, Square)
assert not isinstance(square, Circle)
assert square.area() == 1.0
assert square.perimeter() == 4.0

circle = Shape(type="circle", radius=1.0)
assert type(circle) is Circle
assert isinstance(circle, Shape)
assert not isinstance(circle, Polygon)
assert not isinstance(circle, Rectangle)
assert not isinstance(circle, Square)
assert isinstance(circle, Circle)
assert round(circle.area(), 2) == 3.14
assert round(circle.perimeter(), 2) == 6.28

ValidationError: 1 validation error for RootModel[Annotated[Union[Circle, Square, Rectangle], FieldInfo(annotation=NoneType, required=True, discriminator='type')]]
  Input tag 'polygon' found using 'type' does not match any of the expected tags: 'circle', 'square', 'rectangle' [type=union_tag_invalid, input_value={'type': 'polygon'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/union_tag_invalid

In [ ]:
class Test(type(None)):
    pass

TypeError: type 'NoneType' is not an acceptable base type